In [ ]:
def generate_and_send_payslips():

    import pandas as pd
    import os
    import shutil
    import smtplib
    import win32com.client
    from openpyxl import load_workbook
    from email.message import EmailMessage
    from datetime import datetime
    import time

    print("=== PAYSLIP AUTOMATION SYSTEM ===")

    template = input("Enter path to Payslip Template Excel file: ")
    payroll_file = input("Enter path to Payroll Excel file: ")
    sheet_name = input("Enter sheet name containing payroll data: ")

    excel_folder = input("Enter folder to save Excel payslips: ")
    pdf_folder = input("Enter folder to save PDF payslips: ")
    email_log = input("Enter path to email log CSV file: ")

    sender_email = input("Enter sender Gmail address: ")
    password = input("Enter Gmail App Password: ")

    cc_input = input("Enter CC emails separated by commas: ")
    copy_emails = [email.strip() for email in cc_input.split(",")]

    os.makedirs(excel_folder, exist_ok=True)
    os.makedirs(pdf_folder, exist_ok=True)

    df = pd.read_excel(payroll_file, sheet_name=sheet_name)

    # start Excel
    excel = win32com.client.DispatchEx("Excel.Application")
    excel.Visible = False
    excel.DisplayAlerts = False

    # connect to SMTP
    smtp = smtplib.SMTP_SSL("smtp.gmail.com", 465)
    smtp.login(sender_email, password)

    # create log file if not exists
    if not os.path.exists(email_log):
        with open(email_log, "w") as f:
            f.write("timestamp,employee_id,name,email,status\n")

    for _, row in df.iterrows():

        if pd.isna(row["EMAIL-ADDRESS"]):
            continue

        name = row["Personnel Name"]

        excel_file = os.path.abspath(f"{excel_folder}/Payslip_{name}.xlsx")
        pdf_file = os.path.abspath(f"{pdf_folder}/Payslip_{name}.pdf")

        shutil.copy(template, excel_file)

        wb = load_workbook(excel_file)
        ws = wb.active

        ws['B3'] = row['Personnel Name']
        ws['B4'] = row['Employee Id']
        ws['B5'] = row['Job Title']

        ws['B7'] = row['Total Days of the Month']
        ws['B8'] = row['Days Worked']
        ws['B9'] = row['Total Days of the Month'] - row['Days Worked']
        ws['B11'] = row['New Base Fees']

        ws['B16'] = row['Service Fees (a1)']
        ws['B17'] = row['Overtime/Weekend Work (a3)']
        ws['B18'] = row['Public Holiday']
        ws['B19'] = row['OTHER ALLOWANCE']
        ws['B21'] = row['Transport (a2)']
        ws['B22'] = row['Omitted overtime']
        ws['B23'] = row['Omitted reimbursable']
        ws['B26'] = row['Gross Service Fees']

        ws['E16'] = row['5% WHT']
        ws['E17'] = row['Deduction']
        ws['E26'] = row['Net Pay']

        wb.save(excel_file)

        # convert to PDF
        workbook = excel.Workbooks.Open(excel_file)
        workbook.ExportAsFixedFormat(0, pdf_file)
        workbook.Close(False)

        msg = EmailMessage()

        msg["Subject"] = "Monthly Payslip"
        msg["From"] = sender_email

        emails = [e.strip() for e in str(row["EMAIL-ADDRESS"]).split(";")]
        msg["To"] = ";".join(emails)

        msg["Cc"] = ";".join(copy_emails)

        msg.set_content(f"""
Hello {name},

Please find attached your monthly payslip.

Regards
Finance Department
""")

        with open(pdf_file, "rb") as f:
            msg.add_attachment(
                f.read(),
                maintype="application",
                subtype="pdf",
                filename=os.path.basename(pdf_file)
            )

        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        try:

            smtp.send_message(msg)
            status = "SENT"

        except smtplib.SMTPServerDisconnected:

            smtp = smtplib.SMTP_SSL("smtp.gmail.com", 465)
            smtp.login(sender_email, password)

            smtp.send_message(msg)
            status = "SENT"

        except Exception as e:

            status = f"FAILED - {str(e)}"

        time.sleep(5)

        with open(email_log, "a") as f:
            f.write(f"{timestamp},{row['Employee Id']},{name},{row['EMAIL-ADDRESS']},{status}\n")

    smtp.quit()
    excel.Quit()

    print("Payslip generation and email process completed.")

In [ ]:
generate_and_send_payslips()